# Sensitivity Labels Audit

Audits Microsoft Purview sensitivity label coverage across two surfaces:

| Surface | API used | What you get |
|---|---|---|
| **M365 Groups / Teams** | Graph `assignedLabels` | Which groups have labels, which don't, label distribution |
| **Files (SharePoint / OneDrive)** | Graph `extractSensitivityLabels` | Labels on individual files sampled from site drives |
| **Retention labels** | Graph `retention_label` | Retention policy application on files |

> **Requires:** app registration with `Group.Read.All`, `Sites.Read.All`, `Files.Read.All`
> **Install:** `pip install Office365-REST-Python-Client[notebooks]`
> **Docs:**
> - [Sensitivity labels — Microsoft Purview](https://learn.microsoft.com/en-us/microsoft-365/compliance/sensitivity-labels)
> - [assignedLabels (Graph)](https://learn.microsoft.com/en-us/graph/api/resources/assignedlabel)
> - [extractSensitivityLabels (Graph)](https://learn.microsoft.com/en-us/graph/api/driveitem-extractsensitivitylabels)
> - [retentionLabel (Graph)](https://learn.microsoft.com/en-us/graph/api/driveitem-list-retentionlabel)

> **Scope:** covers label *assignment* (what is applied where).
> Enumerating label *definitions* (the policy catalog) requires
> `GET /beta/informationProtection/policy/labels` — a Purview beta endpoint
> not yet implemented in this library. Label IDs returned here can be
> cross-referenced in the Microsoft Purview compliance portal.

---


## 0 · Setup

In [3]:
from __future__ import annotations

import warnings
from datetime import datetime, timezone

import IPython.display
import pandas as pd
from IPython.display import Markdown
from office365.graph_client import GraphClient
from tests import test_client_id, test_client_secret, test_tenant

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 500)

# ── Auth ───────────────────────────────────────────────────────────────────────
graph = GraphClient(tenant=test_tenant).with_client_secret(test_client_id, test_client_secret)

# Per-file API calls — keep low for large tenants
FILE_SAMPLE_SIZE = 20
DRIVE_SAMPLE_SIZE = 5

print("Tenant:     ", test_tenant)
print("Audit time: ", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))

Tenant:      mediadev8.onmicrosoft.com
Audit time:  2026-05-24 08:45 UTC


---
## 1 · M365 Group Sensitivity Label Coverage

Every M365 Group (and Team) should carry a sensitivity label that governs its
privacy setting, guest access policy, and external sharing. Groups without a
label fall back to tenant defaults — often the most permissive setting.

`assigned_labels` returns `[{labelId, displayName}]` pairs. An empty list
means the group has no label. `classification` is the legacy alternative
(pre-Purview); if present without `assigned_labels` the tenant has not
migrated to unified labeling.


In [2]:
groups = (
    graph.groups.select(["id", "displayName", "assignedLabels", "classification", "visibility", "groupTypes", "mail"])
    .get_all()
    .execute_query()
)

# M365 Groups only (Unified = Teams / SharePoint-backed)
m365_groups = [g for g in groups if "Unified" in (g.properties.get("groupTypes") or [])]

rows = []
for g in m365_groups:
    labels = g.assigned_labels
    label_names = (
        ", ".join(lbl.properties.get("displayName", lbl.properties.get("labelId", "?")) for lbl in labels)
        if labels
        else ""
    )
    label_ids = ", ".join(lbl.labelId for lbl in labels) if labels else ""

    rows.append(
        {
            "Group": g.properties.get("displayName"),
            "Visibility": g.properties.get("visibility", ""),
            "Label(s)": label_names or "None",
            "Label ID(s)": label_ids,
            "Classification": g.properties.get("classification") or "",
            "Email": g.properties.get("mail", ""),
            "_unlabeled": not bool(label_names),
        }
    )

df_groups = pd.DataFrame(rows)
labeled = df_groups[~df_groups["_unlabeled"]]
unlabeled = df_groups[df_groups["_unlabeled"]]

print("M365 Groups total:  ", len(df_groups))
print("With label:         ", len(labeled))
print("Without label:      ", len(unlabeled))
if len(df_groups):
    pct = round(len(labeled) / len(df_groups) * 100, 1)
    print("Coverage:           ", str(pct) + "%")

display(df_groups.drop(columns=["_unlabeled"]))

Unhandled exception in listener <bound method GraphRequest.authenticate_request of <office365.graph_request.GraphRequest object at 0x795b962cb210>>
Traceback (most recent call last):
  File "/home/vgrem/dev/office365-rest-python-client/office365/runtime/types/event_handler.py", line 100, in notify
    listener(*args, **kwargs)
  File "/home/vgrem/dev/office365-rest-python-client/office365/graph_request.py", line 120, in authenticate_request
    token = self._auth_context.acquire_token()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vgrem/dev/office365-rest-python-client/office365/runtime/auth/entra/authentication_context.py", line 42, in acquire_token
    token = TokenResponse.from_json(token_resp)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vgrem/dev/office365-rest-python-client/office365/runtime/auth/token_response.py", line 61, in from_json
    raise ValueError(value)
ValueError: {'error': 'invalid_client', 'error_description': "AADSTS7000215: Invali

ClientRequestException: ('InvalidAuthenticationToken', 'Access token is empty.', '401 Client Error: Unauthorized for url: https://graph.microsoft.com/v1.0/groups?$select=id,displayName,assignedLabels,classification,visibility,groupTypes,mail')

---
## 2 · Label Distribution Across Groups

Which labels are in use and how frequently. A large `None` share indicates
the labeling rollout is incomplete.


In [ ]:
label_counts = df_groups["Label(s)"].value_counts().reset_index()
label_counts.columns = ["Label", "Groups"]
label_counts["% of total"] = (label_counts["Groups"] / len(df_groups) * 100).round(1)
display(label_counts)

---
## 3 · Unlabeled Groups — Governance Gap

Groups without a sensitivity label. Public groups without a label are highest
risk — they inherit tenant defaults and may allow guest access that a label
policy would otherwise restrict.

**Remediation:** `PATCH /groups/{id}` with `{"assignedLabels": [{"labelId": "<guid>"}]}`
Or bulk-assign via the Purview compliance portal.


In [ ]:
public_unlabeled = unlabeled[unlabeled["Visibility"] == "Public"]

print("Unlabeled groups:               ", len(unlabeled))
print("Unlabeled AND public visibility:", len(public_unlabeled), " <- highest risk")

if len(unlabeled):
    IPython.display.display(Markdown("### Groups without sensitivity label"))
    display(
        unlabeled[["Group", "Visibility", "Classification", "Email"]]
        .drop(columns=[], errors="ignore")
        .sort_values(["Visibility", "Group"])
    )

---
## 4 · File-level Sensitivity Label Sampling

`extractSensitivityLabels()` is a **per-file Graph API call** — there is no
bulk query endpoint in the current library. This section samples recently
modified files from a selection of SharePoint site drives.

> `FILE_SAMPLE_SIZE x DRIVE_SAMPLE_SIZE` API calls are made sequentially.
> Keep both values low for large tenants; use section 5 for targeted scans.


In [ ]:
sites = graph.sites.select(["id", "displayName", "webUrl"]).top(DRIVE_SAMPLE_SIZE).get().execute_query()

file_rows = []
skipped = 0

for site in sites:
    site_name = site.properties.get("displayName", site.properties.get("id"))
    try:
        drive = site.drive.get().execute_query()
        items = (
            drive.root.children.select(["id", "name", "webUrl", "lastModifiedDateTime", "file"])
            .top(FILE_SAMPLE_SIZE)
            .get()
            .execute_query()
        )
        for item in items:
            if not item.properties.get("file"):
                continue
            try:
                result = item.extract_sensitivity_labels().execute_query()
                labels = result.value.labels
                file_rows.append(
                    {
                        "Site": site_name,
                        "File": item.properties.get("name"),
                        "Label ID(s)": ", ".join(lbl.sensitivityLabelId for lbl in labels) if labels else "None",
                        "Method": ", ".join(lbl.assignmentMethod for lbl in labels) if labels else "",
                        "Modified": str(item.properties.get("lastModifiedDateTime", ""))[:10],
                    }
                )
            except Exception:
                skipped += 1
    except Exception as exc:
        print("Skipped drive for", site_name, ":", exc)

df_files = pd.DataFrame(file_rows)
labeled_files = df_files[df_files["Label ID(s)"] != "None"] if len(df_files) else df_files
unlabeled_files = df_files[df_files["Label ID(s)"] == "None"] if len(df_files) else df_files

print("Files sampled:   ", len(df_files))
print("With label:      ", len(labeled_files))
print("Without label:   ", len(unlabeled_files))
print("Skipped (error): ", skipped, " (typically protected/system files)")
if len(df_files):
    pct = round(len(labeled_files) / len(df_files) * 100, 1)
    print("File label coverage:", str(pct) + "%")
    display(df_files)

---
## 5 · Targeted Drive Scan

Deep scan of a specific SharePoint document library. Use for Legal, Finance,
HR, or any site known to contain sensitive content.

Edit `TARGET_SITE_URL` and re-run.


In [ ]:
TARGET_SITE_URL = "contoso.sharepoint.com:/sites/Legal"  # <- change this

try:
    target_site = graph.sites.get_by_url(TARGET_SITE_URL).execute_query()
    target_drive = target_site.drive.get().execute_query()
    items = (
        target_drive.root.children.select(["id", "name", "webUrl", "lastModifiedDateTime", "file"])
        .top(50)
        .get()
        .execute_query()
    )
    scan_rows = []
    for item in items:
        if not item.properties.get("file"):
            continue
        try:
            result = item.extract_sensitivity_labels().execute_query()
            labels = result.value.labels
            scan_rows.append(
                {
                    "File": item.properties.get("name"),
                    "Label ID(s)": ", ".join(lbl.sensitivityLabelId for lbl in labels) if labels else "None",
                    "Method": ", ".join(lbl.assignmentMethod for lbl in labels) if labels else "",
                    "Modified": str(item.properties.get("lastModifiedDateTime", ""))[:10],
                }
            )
        except Exception:
            pass
    df_scan = pd.DataFrame(scan_rows)
    print("Files scanned in", TARGET_SITE_URL, ":", len(df_scan))
    display(df_scan)
except Exception as exc:
    print("Could not access site:", exc)
    print("Update TARGET_SITE_URL to: 'tenant.sharepoint.com:/sites/name'")

---
## 6 · Retention Label Coverage

Retention labels enforce how long content must be kept or when it must be
deleted. `is_label_applied_explicitly = True` means a user applied the label
manually — worth flagging since this can override automated policy.


In [ ]:
retention_rows = []

for site in sites:
    site_name = site.properties.get("displayName", site.properties.get("id"))
    try:
        drive = site.drive.get().execute_query()
        items = (
            drive.root.children.select(["id", "name", "webUrl", "file"])
            .expand(["retentionLabel"])
            .top(FILE_SAMPLE_SIZE)
            .get()
            .execute_query()
        )
        for item in items:
            if not item.properties.get("file"):
                continue
            rl = item.retention_label
            props = rl.properties if (rl and rl.properties) else {}
            applied_by = props.get("labelAppliedBy") or {}
            if isinstance(applied_by, dict):
                applied_by_name = (applied_by.get("user") or {}).get("displayName", "")
            else:
                applied_by_name = str(applied_by)
            retention_rows.append(
                {
                    "Site": site_name,
                    "File": item.properties.get("name"),
                    "Retention label": props.get("name") or "None",
                    "Explicit": "Yes" if props.get("isLabelAppliedExplicitly") else "",
                    "Applied by": applied_by_name,
                    "Applied date": str(props.get("labelAppliedDateTime", ""))[:10],
                }
            )
    except Exception as exc:
        print("Skipped", site_name, ":", exc)

df_retention = (
    pd.DataFrame(retention_rows)
    if retention_rows
    else pd.DataFrame(columns=["Site", "File", "Retention label", "Explicit", "Applied by", "Applied date"])
)
with_retention = df_retention[df_retention["Retention label"] != "None"]
explicit_override = df_retention[df_retention["Explicit"] == "Yes"]

print("Files sampled:              ", len(df_retention))
print("With retention label:       ", len(with_retention))
print("Manually applied overrides: ", len(explicit_override))

display(df_retention)

if len(explicit_override):
    IPython.display.display(Markdown("### Manually applied retention labels — verify intent"))
    display(explicit_override[["Site", "File", "Retention label", "Applied by", "Applied date"]])

---
## 7 · Coverage Summary

> **Remediation resources:**
> - Unlabeled groups: [enforce via sensitivity label policy](https://learn.microsoft.com/en-us/microsoft-365/compliance/sensitivity-labels-teams-groups-sites)
> - Unlabeled files: [auto-labeling policies in Purview](https://learn.microsoft.com/en-us/microsoft-365/compliance/apply-sensitivity-label-automatically)
> - Retention: [configure retention policies](https://learn.microsoft.com/en-us/microsoft-365/compliance/retention-policies)


In [ ]:
def _pct(n, total):
    if not total:
        return "n/a"
    return str(round(n / total * 100, 1)) + "%"


findings = [
    ("M365 Groups audited", len(df_groups), ""),
    ("Groups WITH sensitivity label", len(labeled), _pct(len(labeled), len(df_groups))),
    ("Groups WITHOUT label", len(unlabeled), _pct(len(unlabeled), len(df_groups))),
    ("Unlabeled + public visibility", len(public_unlabeled), "highest risk"),
    ("Files sampled (sensitivity labels)", len(df_files), ""),
    ("Files WITH sensitivity label", len(labeled_files), _pct(len(labeled_files), len(df_files))),
    ("Files WITHOUT sensitivity label", len(unlabeled_files), _pct(len(unlabeled_files), len(df_files))),
    ("Files sampled (retention labels)", len(df_retention), ""),
    ("Files with retention label", len(with_retention), _pct(len(with_retention), len(df_retention))),
    ("Manually applied retention overrides", len(explicit_override), "verify intent"),
]

df_summary = pd.DataFrame(findings, columns=["Metric", "Count", "Note"])
display(df_summary.style.hide(axis="index"))

ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
print("Audit completed:", ts)
print()
print("Label IDs can be resolved in the Purview compliance portal:")
print("  https://compliance.microsoft.com")